In [2]:
import numpy as np
import pandas as pd

from contexttimer import Timer
from functools import wraps

In [3]:
def benchmark(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        with Timer() as t:
            result = func(*args, **kwargs)

        print(f"{func.__name__}: {t.elapsed:.4f} seconds")
        return result

    return wrapper

In [4]:
# reproducibility
np.random.seed(42)

# dimensions
n_dates = 1000
n_symbols = 100

dates = pd.date_range("2024-01-01", periods=n_dates)
symbols = [f"SYM{i}" for i in range(n_symbols)]

# multiindex
index = pd.MultiIndex.from_product(
    [dates, symbols],
    names=["date", "symbol"]
)

# dataframe
df = pd.DataFrame(
    {"value": np.random.randn(len(index))},
    index=index
)

df.head(200)

value
date       symbol          
2024-01-01 SYM0    0.496714
           SYM1   -0.138264
           SYM2    0.647689
           SYM3    1.523030
           SYM4   -0.234153
...                     ...
2024-01-02 SYM95   0.385317
           SYM96  -0.883857
           SYM97   0.153725
           SYM98   0.058209
           SYM99  -1.142970

[200 rows x 1 columns]

In [5]:
# randomly remove 10% of rows
drop_indices = np.random.choice(
    df.index,
    size=int(len(df) * 0.1),
    replace=False
)

df_missing = df.drop(drop_indices)

df_missing.head(20)

value
date       symbol          
2024-01-01 SYM0    0.496714
           SYM1   -0.138264
           SYM2    0.647689
           SYM3    1.523030
           SYM4   -0.234153
           SYM5   -0.234137
           SYM6    1.579213
           SYM9    0.542560
           SYM10  -0.463418
           SYM11  -0.465730
           SYM12   0.241962
           SYM14  -1.724918
           SYM15  -0.562288
           SYM17   0.314247
           SYM19  -1.412304
           SYM20   1.465649
           SYM21  -0.225776
           SYM22   0.067528
           SYM23  -1.424748
           SYM25   0.110923

In [6]:
df_missing = df_missing.sort_index()

In [7]:
all_dates = pd.date_range(
    df_missing.index.get_level_values("date").min(),
    df_missing.index.get_level_values("date").max(),
    freq="D"
)

all_symbols = df_missing.index.get_level_values("symbol").unique()

full_index = pd.MultiIndex.from_product(
    [all_dates, all_symbols],
    names=["date", "symbol"]
)

df_missing = df_missing.reindex(full_index)

In [8]:
@benchmark
def loop_ffill(df):
    result = df.copy()

    last_seen = {}

    for idx in result.index:
        symbol = idx[1]

        if pd.isna(result.loc[idx, "value"]):
            if symbol in last_seen:
                result.loc[idx, "value"] = last_seen[symbol]
        else:
            last_seen[symbol] = result.loc[idx, "value"]

    return result

In [9]:
@benchmark
def apply_ffill(df):
    return (
        df.groupby(level="symbol")
        .apply(lambda group: group.ffill())
    )

In [10]:
@benchmark
def groupby_ffill(df):
    return df.groupby(level="symbol").ffill()

In [11]:
@benchmark
def unstack_ffill(df):
    wide = df.unstack(level="symbol")

    filled = wide.ffill()

    return filled.stack()

In [12]:
loop_result = loop_ffill(df_missing)

apply_result = apply_ffill(df_missing)

groupby_result = groupby_ffill(df_missing)

unstack_result = unstack_ffill(df_missing)

loop_ffill: 5.7105 seconds
apply_ffill: 0.0098 seconds
groupby_ffill: 0.0023 seconds
unstack_ffill: 0.0168 seconds


In [13]:
!pip install pyarrow

In [14]:
df_missing.to_parquet("data_single.parquet")

In [15]:
import os

os.listdir()

['var',
 'usr',
 'mnt',
 'etc',
 'dev',
 'run',
 'boot',
 'media',
 'lib',
 'bin',
 'tmp',
 'home',
 'root',
 'opt',
 'sys',
 'proc',
 'sbin',
 'srv',
 'by_symbol',
 '.ipynb_checkpoints',
 'Untitled.ipynb',
 'data_single.parquet',
 'by_date',
 '.dockerenv']

In [16]:
df_loaded = pd.read_parquet("data_single.parquet")

df_loaded.head()

value
date       symbol          
2024-01-01 SYM0    0.496714
           SYM1   -0.138264
           SYM10  -0.463418
           SYM11  -0.465730
           SYM12   0.241962

In [17]:
os.makedirs("by_date", exist_ok=True)

for date, group in df_missing.groupby(level="date"):
    filename = f"by_date/{date.date()}.parquet"

    group.to_parquet(filename)

In [18]:
os.makedirs("by_symbol", exist_ok=True)

for symbol, group in df_missing.groupby(level="symbol"):
    filename = f"by_symbol/{symbol}.parquet"

    group.to_parquet(filename)

In [19]:
@benchmark
def compute_daily_average(df):
    return (
        df.groupby(level="symbol")["value"]
        .mean()
    )

In [20]:
import multiprocessing as mp

In [21]:
def process_symbol(args):
    symbol, group = args

    avg = group["value"].mean()

    return symbol, avg

In [22]:
@benchmark
def multiprocessing_daily_average(df):
    grouped = list(df.groupby(level="symbol"))

    with mp.Pool(processes=mp.cpu_count()) as pool:
        results = pool.map(process_symbol, grouped)

    return pd.Series(dict(results))

In [23]:
from concurrent.futures import ThreadPoolExecutor

In [24]:
@benchmark
def multithreading_daily_average(df):
    grouped = list(df.groupby(level="symbol"))

    with ThreadPoolExecutor() as executor:
        results = executor.map(process_symbol, grouped)

    return pd.Series(dict(results))

In [25]:
!pip install ray

In [26]:
import ray

In [27]:
ray.init()

2026-05-17 15:01:15,515	WARNING services.py:2213 -- WARNING: The object store is using /tmp/ray instead of /dev/shm because /dev/shm has only 67108864 bytes available. This will harm performance! You may be able to free up space by deleting files in /dev/shm. If you are inside a Docker container, you can increase /dev/shm size by passing '--shm-size=2.47gb' to 'docker run' (or add it to the run_options list in a Ray cluster config). Make sure to set this to more than 30% of available RAM.
2026-05-17 15:01:16,642	INFO worker.py:2012 -- Started a local Ray instance.
/opt/conda/lib/python3.13/site-packages/ray/_private/worker.py:2051: FutureWarning: Tip: In future versions of Ray, Ray will no longer override accelerator visible devices env var if num_gpus=0 or num_gpus=None (default). To enable this behavior and turn off this error message, set RAY_ACCEL_ENV_VAR_OVERRIDE_ON_ZERO=0
  warnings.warn(


Python version:,3.13.13
Ray version:,2.55.1


In [28]:
@ray.remote
def ray_process_symbol(symbol, group):
    avg = group["value"].mean()

    return symbol, avg

In [29]:
@benchmark
def ray_daily_average(df):
    grouped = list(df.groupby(level="symbol"))

    futures = [
        ray_process_symbol.remote(symbol, group)
        for symbol, group in grouped
    ]

    results = ray.get(futures)

    return pd.Series(dict(results))

In [30]:
serial_result = compute_daily_average(df_missing)

multiprocessing_result = multiprocessing_daily_average(df_missing)

multithreading_result = multithreading_daily_average(df_missing)

ray_result = ray_daily_average(df_missing)

compute_daily_average: 0.0058 seconds
multiprocessing_daily_average: 0.0906 seconds
multithreading_daily_average: 0.0209 seconds
ray_daily_average: 0.2771 seconds
